# SP-HSPLM Stage 2 — Q9(e) pair-skew cell ladder on H100 / A100

Pre-registered protocol: [`docs/SP_HSPLM_Stage_2_pre-registered_protocol.md`](../../../docs/SP_HSPLM_Stage_2_pre-registered_protocol.md).

This notebook runs one Stage 2 cell per execution. Set the `CELL` constant in the next code cell to one of:

- **Mechanism-2 ladder** (autonomous force law, SPLM "shared across ell" commitment preserved): `q9e_a, q9e_b, q9e_c, q9e_d, q9e_e, q9e_f, q9e_g`.
- **Mechanism-1 extension** (per-layer-indexed force law; lifts the "shared across ell" commitment one submodule at a time, otherwise == `q9e_a`): `q9e_h` (per-layer J_phi), `q9e_i` (per-layer V_phi), `q9e_j` (per-layer alpha_phi), `q9e_k` (joint).
- **Mechanism-1 × Mechanism-2 additivity** (H6 test, EXECUTED 3 seeds; mean PPL 25.64 ± 1.03, median 25.11): `q9e_l` (per-layer J_phi + per-token Omega, shared).
- **Maximal-Mechanism-1** (q9e_l + per-layer Omega): `q9e_m`.
- **Full-Class-F test** (H8; intentionally **non-iso-parameter-count** vs P10g, ~70% more params): `q9e_n` (q9e_m + per-layer V_theta + per-layer V_phi + per-layer alpha_phi). Tests whether the residual ~17 PPL gap to MatchedGPT collapses once V_theta is no longer the autonomous-in-ell bottleneck.

Run all cells; when the cell finishes, change `CELL` and re-run from "Train one cell" downward.

**Architecture is locked to the protocol** for `q9e_a..q9e_m` (the in-class comparison family) — only the C-block schedule, top-`k` routing density, kernel rank `r`, per-token gyro on/off, and the five `share_*_across_layers` flags vary across cells. All other hyperparameters match the SparsePARFLM P10g 16k-step baseline so the H1 PPL comparison is apples-to-apples. **`q9e_n` deliberately breaks iso-parameter-count** (per-layer V_theta dominates the new params, ~10M extra at d=256/v_hidden=1024/v_depth=3); this is the across-class Class-F test, not an in-class comparison.

**Colab bootstrap (automatic).** When run on Google Colab the next cell:

1. mounts your Google Drive at `/content/drive`,
2. shallow-clones the public `dimitarpg13/semsimula` repo into `/content/semsimula`,
3. symlinks the data cache (`notebooks/conservative_arch/data`) to `/content/drive/MyDrive/semsimula_sp_hsplm/data`,
4. routes all training outputs (ckpt, training log, val PPL plot, causal probe history, pair-kernel norms) to `/content/drive/MyDrive/semsimula_sp_hsplm/stage2/{cell}/seed{SEED}/`.

When run locally the notebook walks up from the CWD to find the repo root and writes to `notebooks/conservative_arch/sphsplm/results/sp_hsplm/stage2/{cell}/seed{SEED}/`.

In [ ]:
# ===== Per-run knobs (the user edits these) =====
CELL              = 'q9e_a'                    # one of CELLS below
SEED              = 0
MAX_TRAIN_TOKENS  = 5_000_000

# ===== Repo + Drive layout (hoisted constants, configurable) =====
REPO_URL          = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH       = 'main'
COLAB_REPO_PATH   = '/content/semsimula-paper'        # ephemeral source clone
GDRIVE_OUT_REL    = 'semsimula_sp_hsplm'        # under /content/drive/MyDrive/
RESULTS_DIR_REL   = 'stage2'                    # under GDRIVE_OUT/

CELLS = (
    # Mechanism-2 ladder (autonomous; SPLM "shared across ell")
    'q9e_a', 'q9e_b', 'q9e_c', 'q9e_d', 'q9e_e', 'q9e_f', 'q9e_g',
    # Mechanism-1 extension (per-layer-indexed force law)
    'q9e_h',  # per-layer J_phi
    'q9e_i',  # per-layer V_phi
    'q9e_j',  # per-layer alpha_phi
    'q9e_k',  # per-layer J_phi + V_phi + alpha_phi (joint)
    # Mechanism-1 x Mechanism-2 additivity (H6)
    'q9e_l',  # q9e_d (gyro Omega on, shared) + q9e_h (per-layer J_phi)
    # Maximal-Mechanism-1 (q9e_l + per-layer Omega)
    'q9e_m',  # q9e_l + per-layer Omega^(ell) (L_C independent gyro)
    # Full-Class-F test (H8); intentionally non-iso-parameter-count
    'q9e_n',  # q9e_m + per-layer V_theta + per-layer V_phi + per-layer alpha_phi
)
assert CELL in CELLS, f'CELL must be one of {CELLS}, got {CELL!r}'

import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    """Run a shell command, stream output, raise on non-zero exit."""
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    # ---------- (a) Mount Google Drive ----------
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    # ---------- (b) Shallow-clone (or refresh) the semsimula repo ----------
    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    # ---------- (c) Pin tokenisation cache to Drive (persists across sessions) ----------
    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.exists():
        try:
            repo_data_dir.rmdir()           # only succeeds if empty
        except OSError:
            print(f'NOTE: {repo_data_dir} is non-empty; leaving as-is (no Drive symlink).')
    if not repo_data_dir.exists():
        repo_data_dir.symlink_to(DATA_CACHE, target_is_directory=True)
        print(f'data cache symlink: {repo_data_dir} -> {DATA_CACHE}')

    # ---------- (d) Output root: ckpt / log / plots / probes / norms all live on Drive ----------
    RESULTS_ROOT = GDRIVE_OUT / RESULTS_DIR_REL
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

    # ---------- (e) Make sure the data_module's deps are present ----------
    _sh('pip install -q transformers huggingface_hub pyarrow')

else:
    # ---------- Local / non-Colab: locate repo root by walking up from CWD ----------
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / 'notebooks').exists():
        raise RuntimeError(
            'Could not locate the semsimula repo root from the notebook CWD. '
            'cd into the repo before launching the notebook.'
        )
    RESULTS_ROOT = (
        REPO_ROOT / 'notebooks' / 'conservative_arch' / 'sphsplm'
        / 'results' / 'sp_hsplm' / RESULTS_DIR_REL
    )
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# ===== sys.path so we can import data_module / model_sphsplm / scaleup utils =====
NC_DIR        = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'sphsplm'
PARF_DIR      = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
SCALEUP_DIR   = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
CONS_ARCH_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'

for p in (str(REPO_ROOT), str(CONS_ARCH_DIR), str(SCALEUP_DIR), str(PARF_DIR), str(NC_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'\nREPO_ROOT     = {REPO_ROOT}')
print(f'NC_DIR        = {NC_DIR}')
print(f'PARF_DIR      = {PARF_DIR}')
print(f'SCALEUP_DIR   = {SCALEUP_DIR}')
print(f'RESULTS_ROOT  = {RESULTS_ROOT}')
print(f'CELL = {CELL!r}  SEED = {SEED}')

## Pre-flight — verify environment and the leak invariant

The next two cells:

1. Disable TF32 (mandatory; protocol section 4.1) and set seeds.
2. Verify the data cache is in place and the logfreq surprisal file exists.
3. Run the standalone causal-leak probe on a freshly-instantiated SP-HSPLM model with `causal_force=True`. The expected reading is `max_logit_delta_past = 0.0` exactly. If the reading is non-zero the cell is invalidated and the issue must be diagnosed before training proceeds.

In [ ]:
import torch
import numpy as np

# TF32 is EXPLICITLY DISABLED per Stage 2 protocol section 4.1.
# Three independent knobs are set + asserted + read back:
#   1. torch.backends.cuda.matmul.allow_tf32  = False  (cuBLAS path)
#   2. torch.backends.cudnn.allow_tf32        = False  (cuDNN path)
#   3. torch.set_float32_matmul_precision('highest')   (modern API,
#      equivalent to (1) but explicit at the matmul-precision layer)
# The SPLM/SP-HSPLM autograd.grad forward is sensitive to TF32's 10-bit
# mantissa reduction; running in true fp32 is the protocol's locked
# numerical-stability commitment.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision('highest')
    assert torch.backends.cuda.matmul.allow_tf32 is False
    assert torch.backends.cudnn.allow_tf32 is False
    assert torch.get_float32_matmul_precision() == 'highest'
    print('CUDA available — TF32 explicitly disabled per protocol 4.1')
    print(f'  device                   : {torch.cuda.get_device_name(0)}')
    print(f'  matmul.allow_tf32        : {torch.backends.cuda.matmul.allow_tf32}')
    print(f'  cudnn.allow_tf32         : {torch.backends.cudnn.allow_tf32}')
    print(f'  float32_matmul_precision : {torch.get_float32_matmul_precision()!r}')
else:
    print('CUDA not available; running on CPU (smoke mode only).')
    print(f'  float32_matmul_precision : {torch.get_float32_matmul_precision()!r}  (CPU honours this knob too)')

torch.manual_seed(SEED)
np.random.seed(SEED)

LOGFREQ_NAME = 'logfreq_surprisal_tinystories.npy'
BUNDLED_LOGFREQ = SCALEUP_DIR / 'results' / LOGFREQ_NAME
DRIVE_LOGFREQ   = RESULTS_ROOT / LOGFREQ_NAME
if BUNDLED_LOGFREQ.exists():
    LOGFREQ_PATH = BUNDLED_LOGFREQ
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_PATH = DRIVE_LOGFREQ
else:
    raise FileNotFoundError(
        f'logfreq surprisal cache not found at {BUNDLED_LOGFREQ} or '
        f'{DRIVE_LOGFREQ}. Run scaleup/compute_unigram_frequencies_'
        'tinystories.py first, or copy the cache into the Drive '
        'output dir.'
    )
print(f'logfreq path: {LOGFREQ_PATH}')

In [ ]:
from model_sphsplm import (
    ScalarPotentialLMSPHSPLM, SPHSPLMConfig, _stage2_cell_kwargs,
)

base = dict(
    vocab_size=257, d=16, max_len=64, L=4,
    v_hidden=32, v_depth=2,
    v_phi_d_type=4, v_phi_d_angle=2,
    v_phi_phi_hidden=8, v_phi_theta_hidden=8,
    v_phi_mlp_hidden=16,
    mass_mode='global', ln_after_step=True,
    score_head_hidden=8,
    kernel_rank=4, kernel_init_scale=0.02,
    gyro_rank=4, gyro_init_scale=0.02,
    gamma_min=0.05,
    causal_force=True,
)
kwargs = _stage2_cell_kwargs(CELL, base)
probe_cfg = SPHSPLMConfig(**kwargs)

torch.manual_seed(0)
probe_net = ScalarPotentialLMSPHSPLM(probe_cfg)
probe_net.eval()

T = 32; t_pert = 20
x_a = torch.from_numpy(np.random.default_rng(0).integers(0, probe_cfg.vocab_size, size=(1, T)).astype(np.int64))
x_b = x_a.clone(); x_b[0, t_pert] = (int(x_a[0, t_pert].item()) + 17) % probe_cfg.vocab_size
logits_a = probe_net(x_a)[0].detach()
logits_b = probe_net(x_b)[0].detach()
diffs = (logits_a - logits_b).abs().max(dim=-1).values[0]
pre = float(diffs[:t_pert].max().item())
post = float(diffs[t_pert + 1:].max().item())
print(f'CELL = {CELL}  schedule = {probe_cfg.schedule}')
print(f'  pre-perturbation max |Delta logit| = {pre:.2e}  (expected 0.0)')
print(f'  post-perturbation max |Delta logit| = {post:.2e}  (expected non-zero)')
assert pre == 0.0, f'CAUSAL LEAK at pre-training: pre={pre:.2e}'
assert post > 0.0, f'no post-perturbation effect; model frozen?'
print('  -> leak-clean at init')

## Smoke run (optional, ~2-3 minutes on H100/A100)

Before launching the 16 000-step Stage 2 cell, run a 300-step smoke training to verify the pipeline end-to-end on this machine:

- the data loader works (TinyStories shard 0 loads, the logfreq cache is honoured),
- the SPLM `autograd.grad` forward returns finite gradients at the locked d=256 / L=8 architecture,
- the causal-leak probe stays at exactly `0.0` after the optimiser has had a few hundred steps to find its way into a non-trivial parameter region,
- the pair-kernel norm `||J_phi||_F` does not collapse to machine epsilon in the first 300 steps (early-collapse signal),
- TinyStories validation loss is finite and decreasing (smoke mode reports ppl every 100 steps).

Smoke outputs go to a separate `RESULTS_ROOT/smoke/{cell}/seed{SEED}/` namespace so they cannot collide with the scaleup runs. **Skip this cell if you have already smoke-tested this `CELL` on this machine** — the smoke run takes ~2-3 minutes on H100/A100 but ~30 minutes on CPU. The scaleup cell below is a strict superset of this smoke (same code path, just longer).

In [ ]:
RUN_DIR_SMOKE = RESULTS_ROOT / 'smoke' / CELL / f'seed{SEED}'
RUN_DIR_SMOKE.mkdir(parents=True, exist_ok=True)
print(f'RUN_DIR_SMOKE = {RUN_DIR_SMOKE}')

train_script = NC_DIR / 'train_sphsplm_scaleup.py'
smoke_cmd = [
    sys.executable, str(train_script),
    '--mode', 'smoke',
    '--cell', CELL,
    '--seed', str(SEED),
    '--max-train-tokens', str(MAX_TRAIN_TOKENS),
    '--logfreq-path', str(LOGFREQ_PATH),
    '--results-dir', str(RUN_DIR_SMOKE),
    '--tag-suffix', f'smoke_seed{SEED}',
]
print('\n$ ' + ' '.join(smoke_cmd) + '\n')

# Stream subprocess stdout/stderr line-by-line via Popen so Colab/Jupyter
# captures every line live (subprocess.call writes to raw fd 1 which Colab
# does NOT forward to the cell). Tee to a LOCAL console log on /content/ to
# sidestep the GDrive FUSE write-back lag, plus a copy on Drive at end.
_smoke_console_local = Path('/content') / f'sphsplm_{CELL}_smoke_seed{SEED}_console.log'
try:
    _smoke_console_local.parent.mkdir(parents=True, exist_ok=True)
except Exception:
    _smoke_console_local = Path('.') / _smoke_console_local.name
print(f'(live console log -> {_smoke_console_local})')

_smoke_proc = subprocess.Popen(
    smoke_cmd,
    env={**os.environ, 'OMP_NUM_THREADS': '1', 'KMP_INIT_AT_FORK': 'FALSE',
         'MKL_NUM_THREADS': '1', 'PYTHONUNBUFFERED': '1'},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True,
)
with _smoke_console_local.open('w') as _tee:
    try:
        for _line in _smoke_proc.stdout:
            print(_line, end='', flush=True)
            _tee.write(_line); _tee.flush()
    except KeyboardInterrupt:
        _smoke_proc.terminate(); raise
ret_smoke = _smoke_proc.wait()
print(f'\nsmoke train_sphsplm_scaleup.py exited with code {ret_smoke}')

try:
    _drive_copy = RUN_DIR_SMOKE / _smoke_console_local.name
    shutil.copy2(_smoke_console_local, _drive_copy)
    print(f'console log copied to Drive: {_drive_copy}')
except Exception as _e:
    print(f'(could not copy console log to Drive: {_e})')

assert ret_smoke == 0, f'smoke training failed (exit code {ret_smoke})'

import json as _json
probe_files = sorted(RUN_DIR_SMOKE.glob('*_causal_probe.json'))
norm_files  = sorted(RUN_DIR_SMOKE.glob('*_pair_kernel_norms.json'))
log_files   = sorted(RUN_DIR_SMOKE.glob('*_training_log.jsonl'))

leak_clean = False
init_jphi = float('nan')
final_jphi = float('nan')
final_loss = float('nan')

if probe_files:
    probes = _json.load(open(probe_files[-1]))
    leak_clean = all(p['max_logit_delta_past'] <= 1e-6 for p in probes)
    print(f'\ncausal-leak probe history (smoke):')
    for p in probes:
        verdict = 'clean' if p['max_logit_delta_past'] <= 1e-6 else 'LEAK!'
        print(f"  step {p['step']:>4d} ({p['label']:<5s})  "
              f"max_logit_delta_past = {p['max_logit_delta_past']:.2e}  [{verdict}]")

if norm_files:
    norms = _json.load(open(norm_files[-1]))
    if norms:
        init_jphi  = norms[0]['J_phi_fro']
        final_jphi = norms[-1]['J_phi_fro']

if log_files:
    last_train = None
    with open(log_files[-1]) as f:
        for line in f:
            rec = _json.loads(line)
            if 'train_loss' in rec:
                last_train = rec
    if last_train is not None:
        final_loss = last_train['train_loss']

kernel_active = (
    final_jphi >= 0.05 * init_jphi
    if init_jphi == init_jphi and init_jphi > 0 else False  # NaN-safe
)

print('\n=== smoke verdict ===')
print(f'  cell                       : {CELL}  (seed {SEED})')
print(f'  causal-leak invariant      : {"clean (Delta = 0.0)" if leak_clean else "LEAK!"}')
print(f'  pair-kernel ||J_phi||_F    : init={init_jphi:.4f}  final={final_jphi:.4f}  ratio={final_jphi/max(init_jphi,1e-12):.3f}  ({"active" if kernel_active else "collapsed"})')
print(f'  smoke final train loss     : {final_loss:.4f}')
print(f'  output bundle              : {RUN_DIR_SMOKE}')

assert leak_clean, 'smoke FAIL: causal-leak invariant violated; do not proceed to scaleup'
print('\nsmoke OK — pipeline is healthy; proceed to the scaleup cell below.')

## Train one cell

The next code cell shells out to `train_sphsplm_scaleup.py --mode scaleup --cell {CELL} --seed {SEED}` with `--results-dir` pointing at this notebook's `RUN_DIR`. The training loop:

- runs the SP-HSPLM forward at the locked Stage 2 configuration (16 000 steps, 800 warmup, 800 eval interval, batch 16, block 512, AdamW lr 5e-4 cosine, TF32 disabled),
- anneals the Gumbel temperature linearly from 1.0 to 0.1 over 8 000 steps,
- applies a Frobenius warm-up regulariser to `J_phi` for the first 200 steps,
- runs the causal-leak probe at steps 1, 8 000, 16 000,
- captures pair-kernel norms (`||J_phi||_F`, `||U||_F`, `||V||_F`, plus `||Omega||_F` for q9e_d) at every eval step,
- saves checkpoint, training log, val-PPL plot, causal probe history, pair-kernel norms history, and the cell summary into `RUN_DIR`.

Re-run this notebook (after changing `CELL`) for each of the 7 cells. The dashboard cell at the bottom auto-collects results across whichever cells have completed.

In [ ]:
RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'RUN_DIR = {RUN_DIR}')

train_script = NC_DIR / 'train_sphsplm_scaleup.py'
cmd = [
    sys.executable, str(train_script),
    '--mode', 'scaleup',
    '--cell', CELL,
    '--seed', str(SEED),
    '--max-train-tokens', str(MAX_TRAIN_TOKENS),
    '--logfreq-path', str(LOGFREQ_PATH),
    '--results-dir', str(RUN_DIR),
    '--tag-suffix', f'seed{SEED}',
]
print('\n$ ' + ' '.join(cmd) + '\n')

# Stream subprocess stdout/stderr line-by-line via Popen so Colab/Jupyter
# captures every line live (subprocess.call writes to raw fd 1 which Colab
# does NOT forward to the cell). Tee to a LOCAL console log on /content/ to
# sidestep the GDrive FUSE write-back lag, plus a copy on Drive at end.
_console_local = Path('/content') / f'sphsplm_{CELL}_seed{SEED}_console.log'
try:
    _console_local.parent.mkdir(parents=True, exist_ok=True)
except Exception:
    _console_local = Path('.') / _console_local.name
print(f'(live console log -> {_console_local})')

_proc = subprocess.Popen(
    cmd,
    env={**os.environ, 'OMP_NUM_THREADS': '1', 'KMP_INIT_AT_FORK': 'FALSE',
         'MKL_NUM_THREADS': '1', 'PYTHONUNBUFFERED': '1'},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True,
)
with _console_local.open('w') as _tee:
    try:
        for _line in _proc.stdout:
            print(_line, end='', flush=True)
            _tee.write(_line); _tee.flush()
    except KeyboardInterrupt:
        _proc.terminate(); raise
ret = _proc.wait()
print(f'\ntrain_sphsplm_scaleup.py exited with code {ret}')

try:
    _drive_copy = RUN_DIR / _console_local.name
    shutil.copy2(_console_local, _drive_copy)
    print(f'console log copied to Drive: {_drive_copy}')
except Exception as _e:
    print(f'(could not copy console log to Drive: {_e})')

assert ret == 0, f'training failed (exit code {ret})'

print('\nFiles in RUN_DIR:')
for p in sorted(RUN_DIR.iterdir()):
    print(f'  {p.name}')

## Stage 2 dashboard — collect results across all 7 cells

Auto-collects best/final val PPL, the final pair-kernel Frobenius norm, and the causal-leak verdict from every Stage 2 cell present under `RESULTS_ROOT/{cell}/seed{SEED}/`. Re-run this cell after each cell completes; the dashboard becomes cumulative.

Pre-registered baselines (per protocol section 4.2):

- **SparsePARFLM P10g** (16k steps): val PPL **26.42** — H1 baseline.
- **Stage 1 E4-fix** (solenoidal r=4, 16k steps): val PPL **24.58** — H2 baseline.
- 2σ_seed gate: best Stage 2 cell must beat the baseline by ≥ **3.6 PPL** for clean H1/H2 acceptance.

In [ ]:
import json
import math

P10G_PPL = 26.42
E4_FIX_PPL = 24.58
SIGMA_SEED = 1.8
TWO_SIGMA = 2 * SIGMA_SEED

stage2_results = {}
for cell_name in CELLS:
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        stage2_results[cell_name] = None
        continue
    log_files = sorted(cell_dir.glob('*_training_log.jsonl'))
    probe_files = sorted(cell_dir.glob('*_causal_probe.json'))
    norm_files = sorted(cell_dir.glob('*_pair_kernel_norms.json'))
    if not (log_files and probe_files and norm_files):
        stage2_results[cell_name] = None
        continue
    val_ppls = []
    with log_files[-1].open() as f:
        for line in f:
            rec = json.loads(line)
            if 'val_loss' in rec:
                val_ppls.append(math.exp(rec['val_loss']))
    with probe_files[-1].open() as f:
        probes = json.load(f)
    with norm_files[-1].open() as f:
        norms = json.load(f)
    leak_clean = all(p['max_logit_delta_past'] <= 1e-6 for p in probes)
    final_jphi = norms[-1]['J_phi_fro'] if norms else float('nan')
    init_jphi  = norms[0]['J_phi_fro']  if norms else float('nan')
    kernel_active = (final_jphi >= 0.05 * init_jphi) if init_jphi > 0 else False
    stage2_results[cell_name] = dict(
        best_ppl=min(val_ppls) if val_ppls else float('nan'),
        final_ppl=val_ppls[-1] if val_ppls else float('nan'),
        leak_clean=leak_clean,
        kernel_active=kernel_active,
        init_jphi=init_jphi,
        final_jphi=final_jphi,
    )

print(f'{"cell":<8s} {"best":>7s} {"final":>7s}  {"D vs P10g":>10s}  {"D vs E4":>9s}  {"leak":>6s}  {"kernel":>9s}')
print('-' * 76)
for cell_name, r in stage2_results.items():
    if r is None:
        print(f'{cell_name:<8s} {"--":>7s} {"--":>7s}  {"--":>10s}  {"--":>9s}  {"--":>6s}  {"--":>9s}  (not run)')
        continue
    d_p10g = r["best_ppl"] - P10G_PPL
    d_e4   = r["best_ppl"] - E4_FIX_PPL
    leak_str = "clean" if r["leak_clean"] else "LEAK!"
    kern_str = "active" if r["kernel_active"] else "collapsed"
    print(
        f'{cell_name:<8s} {r["best_ppl"]:7.2f} {r["final_ppl"]:7.2f}  '
        f'{d_p10g:+10.2f}  {d_e4:+9.2f}  {leak_str:>6s}  {kern_str:>9s}'
    )
print('-' * 76)
print(f'P10g baseline (H1): {P10G_PPL:.2f} PPL    E4-fix baseline (H2): {E4_FIX_PPL:.2f} PPL    2*sigma_seed: {TWO_SIGMA:.2f} PPL')

ran = [r for r in stage2_results.values() if r is not None]
if ran:
    best_cell = min(stage2_results.items(), key=lambda kv: kv[1]['best_ppl'] if kv[1] else float('inf'))
    name, info = best_cell
    print(f'\nBest cell: {name}  best PPL = {info["best_ppl"]:.2f}')
    h1_pass = (info['best_ppl'] - P10G_PPL) <= -TWO_SIGMA
    h2_pass = (info['best_ppl'] - E4_FIX_PPL) <= -TWO_SIGMA
    print(f'  H1 (vs P10g, 2-sigma):    {"PASS" if h1_pass else "tie/fail"}')
    print(f'  H2 (vs E4-fix, 2-sigma):  {"PASS" if h2_pass else "tie/fail"}')